# Agentic RAG with Memory using LangGraph ReAct Agent

This notebook extends the simple RAG pipeline by replacing the sequential retrieve-then-generate flow with a ReAct agent that has access to RAG tools and memory management tools. The agent can store, recall, update, and delete user information across conversation sessions using short-term memory (MemorySaver) and long-term memory (InMemoryStore).

## 1. Install Required Libraries

In [2]:
!pip install -q langgraph langchain langchain-openai langchain-community
!pip install -q chromadb
!pip install -q langchain-chroma
!pip install -q pypdf


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Remove ALL preinstalled OpenTelemetry packages
!pip uninstall -y opentelemetry-sdk opentelemetry-api \
    opentelemetry-semantic-conventions opentelemetry-exporter-otlp \
    opentelemetry-exporter-otlp-proto-grpc opentelemetry-proto \
    opentelemetry-instrumentation opentelemetry-instrumentation-logging \
    opentelemetry-instrumentation-threading opentelemetry-instrumentation-asyncio

# Install Patronus and LangChain instrumentation first
!pip install patronus openinference-instrumentation-langchain langchain-mistralai langgraph

# Pin *all* OTel core packages to the version known to work
!pip install --force-reinstall \
    opentelemetry-api==1.37.0 \
    opentelemetry-sdk==1.37.0 \
    opentelemetry-semantic-conventions==0.58b0 \
    opentelemetry-exporter-otlp-proto-grpc==1.37.0 \
    opentelemetry-exporter-otlp==1.37.0 \
    opentelemetry-proto==1.37.0

# Pin instrumentation packages to compatible versions
!pip install --force-reinstall \
    opentelemetry-instrumentation==0.56b0 \
    opentelemetry-instrumentation-logging==0.56b0 \
    opentelemetry-instrumentation-threading==0.56b0 \
    opentelemetry-instrumentation-asyncio==0.56b0

Found existing installation: opentelemetry-sdk 1.37.0
Uninstalling opentelemetry-sdk-1.37.0:
  Successfully uninstalled opentelemetry-sdk-1.37.0
Found existing installation: opentelemetry-api 1.37.0
Uninstalling opentelemetry-api-1.37.0:
  Successfully uninstalled opentelemetry-api-1.37.0
Found existing installation: opentelemetry-semantic-conventions 0.58b0
Uninstalling opentelemetry-semantic-conventions-0.58b0:
  Successfully uninstalled opentelemetry-semantic-conventions-0.58b0
Found existing installation: opentelemetry-exporter-otlp 1.37.0
Uninstalling opentelemetry-exporter-otlp-1.37.0:
  Successfully uninstalled opentelemetry-exporter-otlp-1.37.0
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.37.0
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.37.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.37.0
Found existing installation: opentelemetry-proto 1.37.0
Uninstalling opentelemetry-proto-1.37.0:
  Successfully uninstalled opentel

In [1]:
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry.instrumentation.threading import ThreadingInstrumentor
from opentelemetry.instrumentation.asyncio import AsyncioInstrumentor

import patronus

patronus.init(
    integrations=[
        LangChainInstrumentor(),
        ThreadingInstrumentor(),
        AsyncioInstrumentor(),
    ]
)

PatronusContext(scope=PatronusScope(service='mani-Inspiron-16-7620-2-in-1', project_name='patronus-projects', app='agentic-memory', experiment_id=None, experiment_name=None), tracer_provider=<opentelemetry.sdk.trace.TracerProvider object at 0x74f992c40f10>, logger_provider=<patronus.tracing.logger.LoggerProvider object at 0x74f992c10b90>, api_client_deprecated=<patronus.api.api_client.PatronusAPIClient object at 0x74f992de14d0>, api_client=<patronus_api.PatronusAPI object at 0x74f9a1a99590>, async_api_client=<patronus_api.AsyncPatronusAPI object at 0x74f992dff590>, exporter=<patronus.evals.exporter.BatchEvaluationExporter object at 0x74f992c41c90>, prompts=PromptsConfig(directory=PosixPath('patronus/prompts'), providers=['local', 'api'], templating_engine='f-string'))

## 2. Import Libraries

In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

import warnings
warnings.filterwarnings("ignore")

## 3. Configuration

In [3]:
# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 4. Setup Vector Database

Both company policy documents (leave policies and promotion/bonus policies) are loaded, chunked, and indexed into a single vector database.

### 4.1 Initialize Text Splitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

print("Text splitter initialized!")

Text splitter initialized!


### 4.2 Load and Merge Both Policy Documents

In [5]:
LEAVE_POLICY_PDF_PATH = "/home/mani/PatronusAI/agentic_memory/Company_Leave_Policies_Extended.pdf"
PROMOTION_BONUS_PDF_PATH = "/home/mani/PatronusAI/agentic_memory/Company_Promotion_Bonus_Policies_Enterprise_Grade.pdf"

def load_merged_vectorstore(pdf_paths):
    all_documents = []
    for pdf_path in pdf_paths:
        if not os.path.exists(pdf_path):
            print(f"Warning: PDF not found at {pdf_path}")
            continue
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} pages from {os.path.basename(pdf_path)}")
    
    if not all_documents:
        print("No documents loaded!")
        return None
    
    split_docs = text_splitter.split_documents(all_documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name="company_policies",
        persist_directory="./chroma_db_memory_rag"
    )
    
    print(f"\nTotal: {len(all_documents)} pages, split into {len(split_docs)} chunks")
    return vectorstore

vectorstore = load_merged_vectorstore([LEAVE_POLICY_PDF_PATH, PROMOTION_BONUS_PDF_PATH])
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) if vectorstore else None

Loaded 7 pages from Company_Leave_Policies_Extended.pdf
Loaded 9 pages from Company_Promotion_Bonus_Policies_Enterprise_Grade.pdf

Total: 16 pages, split into 30 chunks


## 5. Define RAG Tool

In [6]:
@tool
def search_company_policies(question: str) -> str:
    """Search company policy documents (leave, promotion, bonus) for relevant information.
    
    Args:
        question: The question to search for in policy documents
    """
    if retriever is None:
        return "No policy documents loaded."
    
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return f"Relevant Policy Information:\n{context}"

## 6. Define Memory Tools (Long-Term Memory)

These tools allow the agent to store, recall, update, and delete information in a persistent memory store that survives across conversation threads.

### 6.1 Initialize Long-Term Memory Store

In [7]:
store = InMemoryStore()
print("Long-term memory store initialized!")

Long-term memory store initialized!


### 6.2 Tool: Save Memory

In [8]:
@tool
def save_memory(key: str, value: str) -> str:
    """Save a piece of information to long-term memory.
    Use this to remember user details, preferences, or any fact
    that should persist across conversations.
    
    Args:
        key: A descriptive key (e.g., 'user_role', 'department', 'preference')
        value: The information to remember
    """
    store.put(("memory",), key, {"content": value})
    return f"Saved to memory: {key} = {value}"

### 6.3 Tool: Recall Memories

In [9]:
@tool
def recall_memories() -> str:
    """Retrieve all stored memories.
    Use this at the start of a conversation to check if there is
    any prior context about the user.
    """
    memories = store.search(("memory",))
    if not memories:
        return "No memories found."
    result = "Stored memories:\n"
    for mem in memories:
        result += f"  {mem.key}: {mem.value['content']}\n"
    return result

### 6.4 Tool: Update Memory

In [10]:
@tool
def update_memory(key: str, new_value: str) -> str:
    """Update an existing memory entry.
    Use this when the user corrects previously stored information,
    such as a role change or updated preference.
    
    Args:
        key: The key of the memory to update
        new_value: The new value to store
    """
    store.put(("memory",), key, {"content": new_value})
    return f"Updated memory: {key} = {new_value}"

### 6.5 Tool: Delete Memory

In [11]:
@tool
def delete_memory(key: str) -> str:
    """Delete a specific memory entry.
    Use this when the user asks you to forget something.
    
    Args:
        key: The key of the memory to delete
    """
    store.delete(("memory",), key)
    return f"Deleted memory: {key}"

## 7. Create Memory-Enabled ReAct Agent

In [12]:
tools = [
    search_company_policies,
    save_memory,
    recall_memories,
    update_memory,
    delete_memory
]

system_prompt = """You are an HR assistant agent with memory capabilities.

Tool Usage Instructions:
1. FIRST use 'recall_memories' to check if there is any stored context
   from prior conversations (e.g., user role, department, preferences).
2. Use 'search_company_policies' to find relevant policy information.
   If you have the user's role/seniority from memory, include it in
   your search query for more targeted results.
3. After answering, use 'save_memory' to store any new facts the user
   shared (e.g., their role, department, seniority level, preferences).
4. If a user corrects previously stored information, use 'update_memory'.
5. If a user asks you to forget something, use 'delete_memory'.

Always check memory first before asking the user to repeat information."""

# Short-term memory: MemorySaver checkpointer (persists within a thread)
checkpointer = MemorySaver()

agent = create_react_agent(llm, tools,
                           prompt=system_prompt,
                           checkpointer=checkpointer,
                           store=store)

print("Memory-enabled ReAct Agent created with 5 tools!")


Memory-enabled ReAct Agent created with 5 tools!


## 8. Query Function

In [13]:
@patronus.traced("agentic-memory-test")
def query_agent(question: str, thread_id: str = "default"):
    """Query the memory-enabled agent.
    
    Args:
        question: The question to ask
        thread_id: Thread ID for short-term conversation history
    """
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke({"messages": [HumanMessage(content=question)]}, config)
    return response['messages'][-1].content

## 9. Test Short-Term Memory (Within a Thread)

Short-term memory is handled by the MemorySaver checkpointer. It tracks all messages within a single conversation thread, so follow-up questions work naturally.

In [14]:
# Turn 1: User shares their role
print("Q1: I am a Senior Software Engineer. How many annual leaves do I have?")
response1 = query_agent("I am a Senior Software Engineer. How many annual leaves do I have?", thread_id="conv-1")
print("A1:", response1)
print("\n" + "="*80 + "\n")

# Turn 2: Follow-up in the same thread (agent remembers Turn 1)
print("Q2: How many of these can I carry forward?")
response2 = query_agent("How many of these can I carry forward?", thread_id="conv-1")
print("A2:", response2)
print("\n" + "="*80 + "\n")

# Turn 3: Another follow-up (agent still has context)
print("Q3: And what about my bonus eligibility?")
response3 = query_agent("And what about my bonus eligibility?", thread_id="conv-1")
print("A3:", response3)

Q1: I am a Senior Software Engineer. How many annual leaves do I have?
A1: As a Senior Software Engineer, you are entitled to 28 days of annual leave. Your leave accrues at a rate of 2.33 days per month, and you can carry forward up to 10 days, with a maximum balance of 38 days.


Q2: How many of these can I carry forward?
A2: You can carry forward up to 10 days of your annual leave.


Q3: And what about my bonus eligibility?
A3: As a Senior Software Engineer, your bonus eligibility can be up to 25% of your annual base salary. The actual bonus payout may range from 0% to 150% of the target, depending on performance outcomes and company results. Bonuses are prorated for partial-year employment.


In [16]:
def print_all_memories():
    # List all namespaces that exist
    namespaces = store.list_namespaces()
    print("Namespaces:", namespaces)

    # Then dump everything
    for ns in namespaces:
        items = store.search(tuple(ns))
        for item in items:
            print(f"  [{'/'.join(ns)}] {item.key}: {item.value}")

print_all_memories()

Namespaces: [('memory',)]
  [memory] user_role: {'content': 'Senior Software Engineer'}


In [17]:
# Print all messages in a specific thread
config = {"configurable": {"thread_id": "conv-1"}}
state = agent.get_state(config)

for msg in state.values["messages"]:
    print(f"[{msg.type}] {msg.content[:200]}")
    print("-" * 80)

[human] I am a Senior Software Engineer. How many annual leaves do I have?
--------------------------------------------------------------------------------
[ai] 
--------------------------------------------------------------------------------
[tool] No memories found.
--------------------------------------------------------------------------------
[ai] 
--------------------------------------------------------------------------------
[tool] Relevant Policy Information:
may impact bonus eligibility.
5.3 Sabbatical Leave (Senior Employees Only)
Senior employees with more than five years of service may apply for sabbatical leave ranging fro
--------------------------------------------------------------------------------
[ai] 
--------------------------------------------------------------------------------
[tool] Saved to memory: user_role = Senior Software Engineer
--------------------------------------------------------------------------------
[ai] As a Senior Software Engineer, you are e

## 11. Test Long-Term Memory (Across Threads)

Long-term memory uses InMemoryStore to persist information across different conversation threads. The agent saves facts using `save_memory` in one thread, then retrieves them in a completely new thread using `recall_memories`.

In [18]:
# Thread 1: User introduces themselves, agent saves to long-term memory
print("Q1: I am a Mid-level employee. Please remember this for future conversations.")
response1 = query_agent("I am a Mid-level employee. Please remember this for future conversations.", thread_id="long-1")
print("A1:", response1)
print("\n" + "="*80 + "\n")

Q1: I am a Mid-level employee. Please remember this for future conversations.
A1: Got it! I've noted that you are a Mid-level employee for future reference. How can I assist you today?




In [21]:
print_all_memories()

Namespaces: [('memory',)]
  [memory] user_role: {'content': 'Senior Software Engineer'}
  [memory] user_seniority: {'content': 'Mid-level'}


In [22]:
# Thread 2: Completely new conversation (different thread id). Agent recalls from long-term memory.
print("Q2: How many days of annual leave do I get?")
response2 = query_agent("How many days of annual leave do I get?", thread_id="long-2")
print("A2 (long-term memory):", response2)
print("\n" + "="*80 + "\n")

Q2: How many days of annual leave do I get?
A2 (long-term memory): You are entitled to 22 days of annual leave per year as a Mid-level Senior Software Engineer.




In [23]:
# Thread 3: Yet another new thread. Agent still knows who the user is.
print("Q3: Am I eligible for a bonus?")
response3 = query_agent("Am I eligible for a bonus?", thread_id="long-3")
print("A3 (long-term memory):", response3)

Q3: Am I eligible for a bonus?
A3 (long-term memory): As a Senior Software Engineer at a mid-level, you are eligible for a bonus of up to 25% of your annual base salary. The actual bonus payout can range from 0% to 150% of the target, depending on performance outcomes and company results. If you have been employed for only part of the year, the bonus will be prorated accordingly.


## 12. Test Memory Update (Handling Contradictions)

When a user corrects information (e.g., they got promoted), the agent updates its long-term memory so all future interactions use the corrected data.

In [24]:
# User reports a promotion
print("Q4: I just got promoted to Senior level employee. Please update my records.")
response4 = query_agent("I just got promoted to Senior level employee. Please update my records.", thread_id="update-1")
print("A4 (memory update):", response4)
print("\n" + "="*80 + "\n")

Q4: I just got promoted to Senior level employee. Please update my records.
A4 (memory update): Congratulations on your promotion! I've updated your records to reflect your new seniority level as a Senior-level employee. If there's anything else you need, feel free to ask!




In [25]:
print_all_memories()

Namespaces: [('memory',)]
  [memory] user_role: {'content': 'Senior Software Engineer'}
  [memory] user_seniority: {'content': 'Senior-level'}


In [ ]:
# New thread: verify the agent uses the updated role
print("Q5: What is my current role and what bonus am I eligible for?")
response5 = query_agent("What is my current role and what bonus am I eligible for?", thread_id="update-2")
print("A5 (After update):", response5)

After update: Your current role is a Senior Software Engineer. As a senior employee, you are eligible for a bonus of up to 25% of your annual base salary. The actual bonus payout can range from 0% to 150% of the target, depending on performance outcomes and company results. Bonuses are also prorated for partial-year employment.


## 13. Test Memory Deletion

The agent can selectively forget information when asked.

In [ ]:
# Ask agent to forget current job role
print("Q6: Please forget my current job role.")
response6 = query_agent("Please forget my current job role.", thread_id="delete-1")
print("A6 (Delete request):", response6)
print("\n" + "="*80 + "\n")

Q1: Please forget my current job role.
A1 (Delete request): I've forgotten your current job role. If there's anything else you need, feel free to let me know!




In [28]:
print_all_memories()

Namespaces: [('memory',)]
  [memory] user_seniority: {'content': 'Senior-level'}


In [ ]:
# Verify: agent should no longer have department in memory
print("Q7: What do you remember about me?")
response7 = query_agent("What do you remember about me?", thread_id="delete-2")
print("A7 (After deletion):", response7)

Q2: What do you remember about me?
A2 (After deletion): I remember that you are at a senior level in your role. Is there anything else you'd like me to remember or update?
